# PPO on SuperAutoStrat

The reusable pieces live in `agents/ppo/`:

| file | what |
|---|---|
| `config.py` | `PPOConfig` hyperparameters, `get_device()` |
| `network.py` | `ActorCritic`: shared trunk, policy logits + value head, masked sampling |
| `envs.py` | `make_vec_env`: N parallel envs (the actors) |
| `rollout.py` | `collect_rollout`: run pi_old for T steps |
| `buffer.py` | `RolloutBuffer`: (T x N) storage + minibatches |
| `advantages.py` | **`compute_gae` is yours to write** (`pytest tests/test_ppo.py` checks it) |
| `losses.py` | `ppo_clip_loss`; **`value_loss`, `entropy_bonus` are yours** |
| `tracking.py` | `EpisodeTracker` for logging |

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn

from agents.ppo import (
    ActorCritic, EpisodeTracker, PPOConfig, RolloutBuffer, collect_rollout, compute_gae,
    entropy_bonus, get_device, make_vec_env, ppo_clip_loss, value_loss,
)

cfg = PPOConfig()
device = get_device()
print(f"Using {device} device")

Using cpu device


# Parallel actors

`N = cfg.n_envs` environments, each in its own process, stepped together. `obs` is `(N, n_states)` and `mask` is `(N, n_actions)`.

In [2]:
envs, obs, mask = make_vec_env(cfg.n_envs, async_envs=cfg.async_envs, seed=0)

n_states = envs.single_observation_space.shape[0]
n_actions = int(envs.single_action_space.n)
obs.shape, mask.shape

((8, 1146), (8, 132))

# Make the Policy Exist

In [3]:
model = ActorCritic(n_states, n_actions).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

# Make the Surrogate Objectives Exist

Sanity check that `ppo_clip_loss` works as an ordinary loss for Adam: build a loss on a fake batch, backprop, step, and watch it go down. (Advantages are random here; real ones come from GAE.)

In [4]:
obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
mask_t = torch.as_tensor(mask, dtype=torch.bool, device=device)

with torch.no_grad():                       # "old" policy: constants
    dist, _ = model.distribution(obs_t, mask_t)
    actions = dist.sample()
    old_log_probs = dist.log_prob(actions)
fake_adv = torch.randn(cfg.n_envs, device=device)

for step in range(3):
    dist, _ = model.distribution(obs_t, mask_t)   # current theta: carries grads
    loss = ppo_clip_loss(dist.log_prob(actions), old_log_probs, fake_adv, cfg.clip_eps)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(step, loss.item())

model = ActorCritic(n_states, n_actions).to(device)  # throw away the fake update
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

0 0.17698460817337036
1 0.15729093551635742
2 0.14014558494091034


# Run Proximal Policy Optimization

In [5]:
buffer = RolloutBuffer(cfg.n_steps, cfg.n_envs, n_states, n_actions, device)
tracker = EpisodeTracker(cfg.n_envs)

for update in range(cfg.n_updates):
    # 1. Run pi_old in N envs for T steps
    obs, mask, last_value = collect_rollout(envs, model, buffer, obs, mask, device, tracker)

    # 2. Compute advantage estimates A_1..A_T   (TODO: advantages.compute_gae)
    # buffer.advantages, buffer.returns = compute_gae(
    #     buffer.rewards, buffer.values, buffer.dones, last_value, cfg.gamma, cfg.gae_lambda)

    # 3. Optimise surrogate L wrt theta, K epochs, minibatch size M <= N*T
    # for epoch in range(cfg.n_epochs):
    #     for mb in buffer.minibatches(cfg.minibatch_size):
    #         TODO: dist, value = model.distribution(mb["obs"], mb["masks"])
    #         TODO: loss = ppo_clip_loss(...) + c1 * value_loss(...) - c2 * entropy_bonus(...)
    #         optimizer.zero_grad()
    #         loss.backward()
    #         nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
    #         optimizer.step()

    # 4. theta_old <- theta: nothing to copy, the next rollout just uses the updated model.

    episodes = tracker.pop()
    if episodes:
        rets, lens = zip(*episodes)
        print(f"update {update}: {len(episodes)} episodes, "
              f"mean return {sum(rets)/len(rets):.2f}, mean length {sum(lens)/len(lens):.0f}")

    if update == 2:
        break  # remove once step 3 is implemented

update 0: 6 episodes, mean return -3.67, mean length 69
update 1: 9 episodes, mean return 0.33, mean length 120
update 2: 10 episodes, mean return 0.50, mean length 114


In [6]:
envs.close()